In [ ]:
# Full-Scale Inference & Data Generation Pipeline

**Objective:**
This notebook utilizes the fine-tuned **Gemma 4 (26B)** model to ingest raw `item_name` strings and autonomously generate full 3-tier taxonomy predictions. 

**Execution Options:**
Based on the 80/10/10 split established during training, you have two options for data generation:
1. **10% Test Set:** Generates predictions strictly on the 10% of data the model was *never* trained on. This is the industry standard for verifying true model intelligence.
2. **100% Full Dataset:** Runs the categorization pipeline across the entire historical database for mass unification.

*Note: This script includes automatic checkpointing. If the cloud instance shuts down unexpectedly, restarting the script will automatically resume from the last saved index.*

In [ ]:
# Install dependencies
!pip install -q transformers peft torch pandas openpyxl scikit-learn tqdm

import os
import json
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# --- Configuration & File Paths ---
INPUT_DIR = 'path/to/your/output_directory/Split_Files'
EXCEL_PATH = os.path.join(INPUT_DIR, 'Categorized.xlsx')
TAXONOMY_PATH = os.path.join(INPUT_DIR, 'master_taxonomy.json')

# Model Paths
BASE_MODEL_ID = "google/gemma-4-26b"
ADAPTER_PATH = "./aws_weights/gemma4_26b_100_percent" # Path to your trained LoRA weights

# Output Configuration
OUTPUT_PATH = "./aws_outputs/Generated_Categorization_Data.xlsx"
AUTOSAVE_PATH = OUTPUT_PATH.replace(".xlsx", "_AUTOSAVE.xlsx")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

TITLE_COLUMN = "item_name" 

HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"

# --- 1. Load Taxonomy ---
print("Loading taxonomy...")
try:
    with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
        taxonomy = json.load(f)
    print("✅ Taxonomy loaded successfully.")
except Exception as e:
    print(f"❌ Error loading taxonomy: {e}")
    taxonomy = {}

# --- 2. Wake Up Gemma 4 (26B) & Inject Custom Weights ---
print(f"\n🤖 Booting base model and applying fine-tuned weights...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)
# Inject LoRA weights for generation
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("✅ Trained AI Engine is online and ready for mass inference.\n")

In [ ]:
# --- The Inference Pipeline ---

def ask_gemma_category(product: str, options: list, level_name: str, parent_context: str = "") -> str:
    """Creates the exact zero-shot prompt format used during training."""
    options_list_str = "\n".join([f"- {opt}" for opt in options])
    context_str = f"Parent Category Path: {parent_context}\n" if parent_context else ""
    
    system_instruction = (
        "You are a strict classification API.\n"
        "Output ONLY the exact category name from the provided 'Valid Categories' list.\n"
        "NO explanations, NO conversational text, NO punctuation outside of the category name."
    )
    
    user_prompt = (
        f"{context_str}Valid {level_name} Categories:\n{options_list_str}\n\n"
        f"Task: Classify this product into one of the categories above.\n"
        f"Product: \"{product}\"\nCategory:"
    )

    messages = [{"role": "user", "content": f"{system_instruction}\n\n{user_prompt}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=20, 
            temperature=0.01, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return response.replace('"', '').replace("'", "").lstrip('- ').strip()

def classify_product(product: str, taxonomy_tree: dict) -> tuple:
    """Cascades through the 3 hierarchy levels using fuzzy validation matching."""
    cat1, cat2, cat3 = "Other", "Other", "Other"
    
    def find_best_match(prediction, valid_options):
        if not prediction: return None
        pred_clean = prediction.strip().lower()
        for opt in valid_options:
            if str(opt).strip().lower() == pred_clean: return opt
        for opt in valid_options:
            if str(opt).strip().lower() in pred_clean: return opt
        return None

    # L1
    l1_options = list(taxonomy_tree.keys())
    l1_pred = ask_gemma_category(product, l1_options, "Level 1")
    cat1_match = find_best_match(l1_pred, l1_options)
    
    if cat1_match:
        cat1 = cat1_match
        # L2
        l2_data = taxonomy_tree[cat1]
        l2_options = list(l2_data.keys()) if isinstance(l2_data, dict) else l2_data
        l2_pred = ask_gemma_category(product, l2_options, "Level 2", cat1)
        cat2_match = find_best_match(l2_pred, l2_options)
        
        if cat2_match and isinstance(l2_data, dict):
            cat2 = cat2_match
            # L3
            l3_data = l2_data[cat2]
            if isinstance(l3_data, (dict, list)):
                l3_options = list(l3_data.keys()) if isinstance(l3_data, dict) else l3_data
                if len(l3_options) > 0:
                    parent_path = f"{cat1} > {cat2}"
                    l3_pred = ask_gemma_category(product, l3_options, "Level 3", parent_path)
                    cat3_match = find_best_match(l3_pred, l3_options)
                    if cat3_match:
                        cat3 = cat3_match

    return cat1, cat2, cat3

In [ ]:
# --- INSTRUCTION: Run THIS cell if you only want to process the 10% Test Split ---

print("Loading dataset and isolating the 10% Unseen Test Set...")
full_df = pd.read_excel(EXCEL_PATH)

# Reproducing the exact split from the training script
train_df, temp_df = train_test_split(full_df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Set the target dataframe for the automation loop
target_df = test_df.copy()
target_df.reset_index(drop=True, inplace=True)

print(f"✅ Target Data: 10% Test Set ({len(target_df)} rows).")

In [ ]:
# --- INSTRUCTION: Run THIS cell if you want to process the entire dataset ---

print("Loading 100% Full Dataset...")
full_df = pd.read_excel(EXCEL_PATH)

# Set the target dataframe for the automation loop
target_df = full_df.copy()

print(f"✅ Target Data: 100% Full Dataset ({len(target_df)} rows).")

In [ ]:
# --- Resilient Automation Loop (Auto-Save & Resume) ---

results_list = []
start_index = 0

# 1. Check for crash recovery
if os.path.exists(AUTOSAVE_PATH):
    print(f"🔄 Found existing autosave. Recovering previously predicted data...")
    try:
        prev_df = pd.read_excel(AUTOSAVE_PATH)
        results_list = prev_df.to_dict('records')
        start_index = len(results_list)
        print(f"✅ Successfully loaded {start_index} previous predictions.")
    except Exception as e:
        print(f"⚠️ Warning: Could not load previous autosave: {e}")

# 2. Slice dataframe to skip completed rows
df_to_process = target_df.iloc[start_index:]

print(f"\n🚀 Beginning Mass Categorization Engine...")
print(f"Remaining products to classify: {len(df_to_process)}\n")

# 3. Execution Loop
for count, (original_idx, row) in enumerate(tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Classifying"), start_index + 1):
    product_name = str(row.get(TITLE_COLUMN, "Unknown Product"))
    
    # AI Inference
    c1, c2, c3 = classify_product(product_name, taxonomy)
    
    # Store Output
    results_list.append({
        "item_name": product_name,
        "category_1_pred": c1,
        "category_2_pred": c2,
        "category_3_pred": c3
    })
    
    # Autosave Trigger (Every 1000 items)
    if count % 1000 == 0:
        pd.DataFrame(results_list).to_excel(AUTOSAVE_PATH, index=False)
        tqdm.write(f"💾 Checkpoint: Auto-saved {count} items to '{AUTOSAVE_PATH}'.")

# 4. Final Export
print("\n\n✅ Mass Categorization Finished! Compiling final dataset...")
final_output_df = pd.DataFrame(results_list)
final_output_df.to_excel(OUTPUT_PATH, index=False)

if os.path.exists(AUTOSAVE_PATH):
    os.remove(AUTOSAVE_PATH)
    print("🧹 Cleaned up temporary autosave files.")

print(f"🎉 Complete! Check your final dataset at:\n{OUTPUT_PATH}")